# Compare Legacy And API Overlap RF Models

Ноутбук сравнивает две Random Forest модели на одном и том же USA train/test split:

- legacy модель на признаках `['d', 'm_o', 'm_d']`
- overlap модель на `API_OVERLAP_FEATURE_COLS`

Сравнение идёт по тем же метрикам, которые уже используются в проекте: `CPC`, `ACC`, `RMSE`, `RE`, `LR`.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
MIGRATION_DIR = PROJECT_ROOT / 'migration'

if str(MIGRATION_DIR) not in sys.path:
    sys.path.insert(0, str(MIGRATION_DIR))

from config import (
    DATA_PATH,
    FEATURE_COLS,
    API_OVERLAP_FEATURE_COLS,
    MODEL_PATH,
    API_OVERLAP_MODEL_PATH,
)
from data_loading import load_datasets
from model_predictions import load_model
from model_training import evaluate_model, train_and_save_model


In [2]:
RETRAIN_LEGACY_MODEL = True
RETRAIN_OVERLAP_MODEL = True
N_ESTIMATORS = 200
RANDOM_STATE = 42
VERBOSE = 1
TRAIN_VARIANT = "full"

LEGACY_MODEL_PATH = Path(MODEL_PATH)
OVERLAP_MODEL_PATH = Path(API_OVERLAP_MODEL_PATH)

print(f'TRAIN_VARIANT = {TRAIN_VARIANT}')
print(f'RETRAIN_LEGACY_MODEL = {RETRAIN_LEGACY_MODEL}')
print(f'RETRAIN_OVERLAP_MODEL = {RETRAIN_OVERLAP_MODEL}')
print(f'N_ESTIMATORS = {N_ESTIMATORS}')
print(f'RANDOM_STATE = {RANDOM_STATE}')
print(f'LEGACY_MODEL_PATH = {LEGACY_MODEL_PATH}')
print(f'OVERLAP_MODEL_PATH = {OVERLAP_MODEL_PATH}')


TRAIN_VARIANT = full
RETRAIN_LEGACY_MODEL = True
RETRAIN_OVERLAP_MODEL = True
N_ESTIMATORS = 200
RANDOM_STATE = 42
LEGACY_MODEL_PATH = models\RF\rf_model.joblib
OVERLAP_MODEL_PATH = models\RF\rf_model_api_overlap.joblib


In [3]:
train_data, test_data, _ = load_datasets(DATA_PATH, train_variant=TRAIN_VARIANT)

states = sorted(train_data.keys())
print(f'States in train split: {states}')
print(f'Train rows total: {sum(len(df) for df in train_data.values())}')
print(f'Test rows total: {sum(len(df) for df in test_data.values())}')


States in train split: ['California', 'Florida', 'Massachusetts', 'New York', 'Texas', 'Washington']
Train rows total: 93264
Test rows total: 30816


In [4]:
def train_or_load_model(model_path, feature_cols, retrain, label):
    model_path = Path(model_path)
    if retrain or not model_path.exists():
        print(f'Training {label} model -> {model_path}')
        model, results = train_and_save_model(
            train_data=train_data,
            test_data=test_data,
            model_path=str(model_path),
            feature_cols=feature_cols,
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            verbose=VERBOSE,
        )
    else:
        print(f'Loading existing {label} model <- {model_path}')
        model = load_model(str(model_path))
        results = evaluate_model(model, test_data, feature_cols=feature_cols)
    return model, results


def normalize_results(results, model_name):
    flat = results.copy()
    if isinstance(flat.index, pd.MultiIndex):
        flat.index = flat.index.get_level_values(-1)
    flat.index.name = 'metric'
    flat.columns.name = 'state'
    flat = flat.T.reset_index().melt(id_vars='state', var_name='metric', value_name='value')
    flat.insert(0, 'model', model_name)
    return flat


In [5]:
legacy_model, legacy_results = train_or_load_model(
    model_path=LEGACY_MODEL_PATH,
    feature_cols=FEATURE_COLS,
    retrain=RETRAIN_LEGACY_MODEL,
    label='legacy',
)

print('Legacy features:')
print(FEATURE_COLS)
display(legacy_results)


Training legacy model -> models\RF\rf_model.joblib


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   14.8s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   16.5s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Do

Legacy features:
['d', 'm_o', 'm_d']


New York  Massachusetts   California      Florida  \
Random Forest CPC      0.495384       0.543335     0.488159     0.449927   
              ACC      0.821112       0.798523     0.769818     0.806465   
              RMSE  1316.181583    2381.974206  6613.125214  2771.907770   
              RE       1.175814       1.263319     0.820649     1.117091   
              LR       0.745972       0.744139     0.672526     0.730862   

                     Washington        Texas  
Random Forest CPC      0.458855     0.473562  
              ACC      0.757069     0.817603  
              RMSE  2896.171172  2890.795326  
              RE       1.062547     0.962605  
              LR       0.747180     0.680746

In [6]:
overlap_model, overlap_results = train_or_load_model(
    model_path=OVERLAP_MODEL_PATH,
    feature_cols=API_OVERLAP_FEATURE_COLS,
    retrain=RETRAIN_OVERLAP_MODEL,
    label='api_overlap',
)

print('API overlap features:')
print(API_OVERLAP_FEATURE_COLS)
display(overlap_results)


Training api_overlap model -> models\RF\rf_model_api_overlap.joblib


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    6.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   32.8s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   37.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Do

API overlap features:
['d', 'area_o', 'area_d', 'm_o', 'm_d', 'health_point_o', 'health_point_d', 'main_road_line_o', 'main_road_line_d', 'school_point_o', 'school_point_d']


New York  Massachusetts   California      Florida  \
Random Forest CPC      0.517465       0.599396     0.491790     0.529802   
              ACC      0.819380       0.821601     0.783150     0.820209   
              RMSE  1213.729629    1811.243811  6663.536847  2411.467384   
              RE       0.900294       0.915653     0.673990     0.957778   
              LR       0.637619       0.623799     0.597306     0.642449   

                     Washington        Texas  
Random Forest CPC      0.577157     0.521218  
              ACC      0.820674     0.838100  
              RMSE  2565.939955  2790.079337  
              RE       1.208410     1.023683  
              LR       0.675693     0.658585

In [7]:
legacy_flat = normalize_results(legacy_results, 'legacy')
overlap_flat = normalize_results(overlap_results, 'api_overlap')
comparison_long = pd.concat([legacy_flat, overlap_flat], ignore_index=True)

comparison_table = comparison_long.pivot_table(
    index=['state', 'metric'],
    columns='model',
    values='value',
)
comparison_table['delta_overlap_minus_legacy'] = comparison_table['api_overlap'] - comparison_table['legacy']
comparison_table = comparison_table.reset_index()

display(comparison_table)


model,state,metric,api_overlap,legacy,delta_overlap_minus_legacy
0,California,ACC,0.783150,0.769818,0.013332
1,California,CPC,0.491790,0.488159,0.003631
2,California,LR,0.597306,0.672526,-0.075221
3,California,RE,0.673990,0.820649,-0.146659
4,California,RMSE,6663.536847,6613.125214,50.411633
5,Florida,ACC,0.820209,0.806465,0.013744
6,Florida,CPC,0.529802,0.449927,0.079875
7,Florida,LR,0.642449,0.730862,-0.088413
8,Florida,RE,0.957778,1.117091,-0.159313
9,Florida,RMSE,2411.467384,2771.907770,-360.440386


In [8]:
summary = comparison_long.pivot_table(
    index='metric',
    columns='model',
    values='value',
    aggfunc='mean',
)
summary['delta_overlap_minus_legacy'] = summary['api_overlap'] - summary['legacy']
metric_direction = {
    'CPC': 'higher',
    'ACC': 'higher',
    'RMSE': 'lower',
    'RE': 'lower',
    'LR': 'lower',
}
summary['preferred_direction'] = summary.index.map(metric_direction)

def choose_better(row):
    if row['preferred_direction'] == 'higher':
        return 'api_overlap' if row['api_overlap'] > row['legacy'] else 'legacy'
    return 'api_overlap' if row['api_overlap'] < row['legacy'] else 'legacy'

summary['better_model'] = summary.apply(choose_better, axis=1)
display(summary)


model,api_overlap,legacy,delta_overlap_minus_legacy,preferred_direction,better_model
metric,,,,,
ACC,0.817186,0.795098,0.022087,higher,api_overlap
CPC,0.539472,0.484870,0.054601,higher,api_overlap
LR,0.639242,0.720238,-0.080996,lower,api_overlap
RE,0.946634,1.067004,-0.120370,lower,api_overlap
RMSE,2909.332827,3145.025879,-235.693051,lower,api_overlap


<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>model</th>
      <th>api_overlap</th>
      <th>legacy</th>
      <th>delta_overlap_minus_legacy</th>
      <th>preferred_direction</th>
      <th>better_model</th>
    </tr>
    <tr>
      <th>metric</th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
    </tr>

  </thead>
  <tbody>
    <tr>
      <th>ACC</th>
      <td>0.817591</td>
      <td>0.795527</td>
      <td>0.022064</td>
      <td>higher</td>
      <td>api_overlap</td>
    </tr>
    <tr>
      <th>CPC</th>
      <td>0.540102</td>
      <td>0.485236</td>
      <td>0.054865</td>
      <td>higher</td>
      <td>api_overlap</td>
    </tr>
    <tr>
      <th>LR</th>
      <td>0.637610</td>
      <td>0.718442</td>
      <td>-0.080832</td>
      <td>lower</td>
      <td>api_overlap</td>
    </tr>
    <tr>
      <th>RE</th>
      <td>0.944299</td>
      <td>1.062951</td>
      <td>-0.118652</td>
      <td>lower</td>
      <td>api_overlap</td>
    </tr>
    <tr>
      <th>RMSE</th>
      <td>2903.233007</td>
      <td>3132.360769</td>
      <td>-229.127762</td>
      <td>lower</td>
      <td>api_overlap</td>
    </tr>
  </tbody>
</table>
</div>

In [9]:
output_dir = PROJECT_ROOT / 'artifacts' / 'model_comparison'
output_dir.mkdir(parents=True, exist_ok=True)

comparison_csv = output_dir / 'legacy_vs_api_overlap_by_state.csv'
summary_csv = output_dir / 'legacy_vs_api_overlap_summary.csv'

comparison_table.to_csv(comparison_csv, index=False)
summary.reset_index().to_csv(summary_csv, index=False)

print(f'Per-state comparison saved to: {comparison_csv}')
print(f'Summary saved to: {summary_csv}')


Per-state comparison saved to: d:\programming\github\Migration-model\artifacts\model_comparison\legacy_vs_api_overlap_by_state.csv
Summary saved to: d:\programming\github\Migration-model\artifacts\model_comparison\legacy_vs_api_overlap_summary.csv
